# Comparing summarizers on CNN/DailyMail

Runs **three** summarization models over `test.csv`, scores each generated
summary against the human `highlights`, and compares them on **three** metrics.

**Models**

| Model | Notes |
|-------|-------|
| `facebook/bart-large-cnn` | BART, fine-tuned on CNN/DailyMail |
| `google/pegasus-cnn_dailymail` | Pegasus, fine-tuned on CNN/DailyMail |
| `allenai/led-large-16384-arxiv` | LED (Longformer Encoder-Decoder), long-doc; fine-tuned on **arXiv**, not CNN/DM |

**Metrics:** ROUGE (1/2/L), BERTScore-F1, METEOR.

> Note: BART and Pegasus are CNN/DailyMail-tuned, so they have a home-field
> advantage here. LED is included to test a long-document model that was tuned
> on a *different* domain (arXiv) — expect lower CNN/DM scores. Swap in a
> different LED/LongT5 checkpoint via the `MODELS` list below.

First run installs packages and downloads each model once (LED and Pegasus are
~2 GB each), so it's slow the first time.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torch", "pandas", "matplotlib",
                "rouge-score", "bert-score", "nltk", "sentencepiece", "protobuf"], check=True)

CompletedProcess(args=['/Users/angie/miniforge3/bin/python', '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'pandas', 'matplotlib', 'rouge-score', 'bert-score', 'nltk', 'sentencepiece', 'protobuf'], returncode=0)

In [2]:
import gc
import pandas as pd
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk import word_tokenize
from bert_score import score as bert_score

# NLTK data needed by METEOR (downloaded once).
for res, path in [("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4"),
                  ("punkt", "tokenizers/punkt"), ("punkt_tab", "tokenizers/punkt_tab")]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(res, quiet=True)

/Users/angie/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [3]:
# Each model: its id and how many input tokens to keep (CNN/DM articles are short,
# so 1024 is plenty for BART/Pegasus; LED can take more but doesn't need to here).
MODELS = [
    {"name": "facebook/bart-large-cnn",       "max_input": 1024},
    {"name": "google/pegasus-cnn_dailymail",  "max_input": 1024},
    {"name": "allenai/led-large-16384-arxiv", "max_input": 4096},
]

TEST_CSV   = "test.csv"
LIMIT      = 200    # None = ALL 11,490 rows (very long, esp. LED). Start small.
BATCH_SIZE = 8      # lower to 4 if you hit out-of-memory (LED is the heaviest)
MAX_LEN, MIN_LEN = 128, 30

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE)

Device: mps


## Load the test set

In [5]:
df = pd.read_csv(TEST_CSV).dropna(subset=["article", "highlights"])
df = df[(df["article"].str.strip() != "") & (df["highlights"].str.strip() != "")]
if LIMIT:
    df = df.head(LIMIT)
df = df.reset_index(drop=True)
articles, references = df["article"].tolist(), df["highlights"].tolist()
print(f"Evaluating {len(df)} examples across {len(MODELS)} models")

Evaluating 200 examples across 3 models


## Generation + scoring helpers

`AutoModelForSeq2SeqLM.generate()` works for all three (avoids the
`pipeline("summarization")` task that newer transformers dropped). LED needs a
global-attention mask on the first token; Pegasus emits `<n>` separators that we
normalise for fair scoring.

In [6]:
def is_led(name):
    return "led" in name.lower()

def generate_summaries(cfg):
    name = cfg["name"]
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
    preds = []
    for start in range(0, len(articles), BATCH_SIZE):
        inputs = tokenizer(articles[start:start + BATCH_SIZE], max_length=cfg["max_input"],
                           truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        gen_kwargs = dict(max_length=MAX_LEN, min_length=MIN_LEN, num_beams=4, do_sample=False)
        if is_led(name):
            gmask = torch.zeros_like(inputs["input_ids"])
            gmask[:, 0] = 1  # LED: global attention on the first token
            gen_kwargs["global_attention_mask"] = gmask
        with torch.no_grad():
            ids = model.generate(**inputs, **gen_kwargs)
        preds.extend(t.replace("<n>", " ").strip() for t in tokenizer.batch_decode(ids, skip_special_tokens=True))
        print(f"  [{name}] {min(start + BATCH_SIZE, len(articles))}/{len(articles)}", end="\r")
    print()
    # free memory before loading the next model
    del model
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    return preds

_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluate(preds, refs):
    r1 = r2 = rL = met = 0.0
    for p, r in zip(preds, refs):
        sc = _rouge.score(r, p)
        r1 += sc["rouge1"].fmeasure; r2 += sc["rouge2"].fmeasure; rL += sc["rougeL"].fmeasure
        met += meteor_score([word_tokenize(r)], word_tokenize(p))
    _, _, f1 = bert_score(preds, refs, lang="en", rescale_with_baseline=True)
    n = len(preds)
    return {"rouge1": r1 / n, "rouge2": r2 / n, "rougeL": rL / n,
            "bertscore_f1": f1.mean().item(), "meteor": met / n}

## Run all models

This is the slow cell: it summarizes every article with each model, then scores.

In [ ]:
results = {}
for cfg in MODELS:
    print(f"=== {cfg['name']} ===")
    preds = generate_summaries(cfg)
    results[cfg["name"]] = evaluate(preds, references)
    print("  ", {k: round(v, 4) for k, v in results[cfg["name"]].items()})

=== facebook/bart-large-cnn ===


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 9858.56it/s]


  [facebook/bart-large-cnn] 200/200


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 9991.51it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/angie/miniforge3/lib/python3.13/site-packages/bert_score/score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behav

   {'rouge1': 0.4309, 'rouge2': 0.201, 'rougeL': 0.2959, 'bertscore_f1': 0.2965, 'meteor': 0.3869}
=== google/pegasus-cnn_dailymail ===


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 61533.23it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Comparison

In [ ]:
table = pd.DataFrame(results).T[["rouge1", "rouge2", "rougeL", "bertscore_f1", "meteor"]].round(4)
table

In [ ]:
import matplotlib.pyplot as plt

ax = table.plot(kind="bar", figsize=(10, 5))
ax.set_title(f"Summarizer comparison on CNN/DailyMail test (n={len(references)})")
ax.set_ylabel("score (0–1)")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", title="metric")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()